# MusicBrainz collection (stratified sample)

Runs `musicbrainz_collect.py` on `spotify_stratified_sample.csv`.

- **Rate limit:** at most 1 request/second, plus any `Retry-After` the server sends
- **Cache:** one raw JSON file per track in `data/raw/musicbrainz/recordings/`. Tracks already on disk are skipped, so the run can be stopped and resumed
- **Retries:** 429/5xx/timeouts/connection errors are retried with exponential backoff
- **Logs:** `logs/musicbrainz_collect.log` (readable) and `logs/musicbrainz_manifest.jsonl` (one line per request)

At 1 request/second: 2 per genre ≈ 4 min, 60 per genre (the project plan's 50–80) ≈ 2 h, the full 53,942-track sample ≈ 15 h. For long runs, use the terminal instead: `python musicbrainz_collect.py --per-genre 60`

In [1]:
import importlib, json
import pandas as pd
import musicbrainz_collect as mb
importlib.reload(mb)

PER_GENRE = 60  # project plan: 50-80 per genre (2 = quick test, 0 = whole sample); cached tracks are skipped
stats = mb.collect(per_genre=PER_GENRE or None)
stats

2026-11-18 17:12:02,468 INFO    Start: 6780 tracks (per_genre=60, limit=None, refresh=False)
2026-11-18 17:12:02,983 WARNING   attempt 1 failed (HTTP 503); retrying in 2.7s
2026-11-18 17:12:05,938 INFO    [555/6780] brazil | Marcelo Falcão - Sexta-Feira -> 1 result(s) (top score 100)
2026-11-18 17:12:06,878 WARNING   attempt 1 failed (HTTP 503); retrying in 2.3s
2026-11-18 17:12:09,375 WARNING   attempt 2 failed (HTTP 503); retrying in 4.3s
2026-11-18 17:12:13,875 INFO    [1038/6780] club | Glee Cast - If I Die Young (Glee Cast Version) -> 0 result(s)
2026-11-18 17:12:14,877 INFO    [1072/6780] club | The 6ths - You You You You You -> 2 result(s) (top score 100)
2026-11-18 17:12:15,884 INFO    [2731/6780] guitar | Project AER - Brighter Days -> 1 result(s) (top score 100)
2026-11-18 17:12:17,093 INFO    [4491/6780] mpb | Sandy - Universo Reduzido -> 1 result(s) (top score 100)
2026-11-18 17:12:17,908 INFO    [6296/6780] study | Matt Large - I Can't Breathe -> 1 result(s) (top score 100

{'total': 6780,
 'cached': 6774,
 'fetched': 6,
 'errors': 0,
 'with_results': 5,
 'elapsed_min': 0.3}

In [2]:
# Per-request log: match rate and how confident the top match is
man = pd.read_json(mb.MANIFEST, lines=True).drop_duplicates("track_id", keep="last")
print(f"Requests logged: {len(man):,}")
print(man["result"].value_counts(), "\n")
ok = man[man["result"] == "ok"]
print(f"Tracks with >=1 MusicBrainz candidate: {(ok['n_results'] > 0).mean():.1%}")
print(f"Top score == 100: {(ok['top_score'] == 100).mean():.1%}")
print(f"Retried requests: {(man['attempts'] > 1).sum()}")
man.groupby("track_genre")["n_results"].apply(lambda s: (s > 0).mean()).sort_values().head(10).rename("match_rate")

Requests logged: 6,780
result
ok    6780
Name: count, dtype: int64 

Tracks with >=1 MusicBrainz candidate: 62.5%
Top score == 100: 62.5%
Retried requests: 597


track_genre
sleep        0.083333
romance      0.100000
malay        0.116667
classical    0.133333
forro        0.216667
cantopop     0.233333
sertanejo    0.233333
kids         0.266667
gospel       0.283333
pagode       0.300000
Name: match_rate, dtype: float64

In [3]:
# Peek at one stored raw JSON file
f = next(mb.RAW_DIR.glob("*.json"))
rec = json.loads(f.read_text())
print(f.name)
print(json.dumps(rec["spotify"], indent=1, ensure_ascii=False))
top = rec["response"]["recordings"][:1]
print(json.dumps(top, indent=1, ensure_ascii=False)[:1500])

6rUlXC375tM8gsgQy5K110.json
{
 "track_name": "Drums Of War",
 "artists": "Sub Sonik",
 "album_name": "Drums Of War",
 "duration_ms": 151966,
 "track_genre": "hardstyle"
}
[
 {
  "id": "bddada86-8102-4ec2-9efd-e16f15b4fda5",
  "score": 100,
  "artist-credit-id": "ec7b5fca-d0b1-31fc-ae65-30fd0ae749fa",
  "title": "Drums of War",
  "length": 154826,
  "disambiguation": "part of “Hard Bass 2018” DJ‐mix",
  "video": null,
  "artist-credit": [
   {
    "name": "Sub Sonik",
    "artist": {
     "id": "28f0ecf1-b1fb-4421-ac64-ec5b26cbd490",
     "name": "Sub Sonik",
     "sort-name": "Sub Sonik",
     "disambiguation": "Hardstyle dj/producer Ilmar Hansen"
    }
   }
  ],
  "first-release-date": "2018-01-29",
  "releases": [
   {
    "id": "faee3d78-0ad2-4a58-8ae1-7f6051f3ffb6",
    "status-id": "4e304316-386d-3409-af2e-78857eec5cfe",
    "artist-credit-id": "949a7fd5-fe73-3e8f-922e-01ff4ca958f7",
    "count": 4,
    "title": "Hard Bass 2018",
    "status": "Official",
    "disambiguation": "mi